# Verona E-Commerce — Customer Intelligence
## RFM Segmentation & Cohort Retention Analysis

This notebook analyzes the outputs of `src/feature_engineering.py`:
- `data/processed/rfm_segments.csv`
- `data/processed/cohort_retention.csv`

**Business questions answered:** Who are our most valuable customers? Which segments should we prioritize for retention vs. win-back? How well do we retain customers over time after acquisition?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

rfm = pd.read_csv("../data/processed/rfm_segments.csv")
cohort = pd.read_csv("../data/processed/cohort_retention.csv", index_col=0)
rfm.head()


## RFM Methodology (recap)

- **Recency** = days since last completed order (lower = better)
- **Frequency** = count of distinct completed orders
- **Monetary** = total realized revenue
- Each dimension scored 1-5 by quintile; segments assigned using standard RFM segment rules (see `src/feature_engineering.py` docstring for exact thresholds).


In [ ]:
seg_summary = rfm.groupby('segment').agg(
    customers=('customer_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
).sort_values('total_revenue', ascending=False)
seg_summary['pct_of_customers'] = seg_summary['customers'] / seg_summary['customers'].sum()
seg_summary['pct_of_revenue'] = seg_summary['total_revenue'] / seg_summary['total_revenue'].sum()
seg_summary.round(2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
seg_summary['customers'].plot(kind='bar', ax=axes[0], color='#1a3d5c', title='Customers per Segment')
seg_summary['total_revenue'].plot(kind='bar', ax=axes[1], color='#2c5f8a', title='Revenue per Segment')
axes[0].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


**Insight:** Segment size and segment revenue are not proportional — Champions are a minority of customers but generate the majority of revenue, while Lost Customers are numerically large but contribute comparatively little. This is the core justification for differentiated retention spend by segment rather than a one-size-fits-all approach.


In [ ]:
plt.figure(figsize=(8,8))
plt.pie(seg_summary['pct_of_revenue'], labels=seg_summary.index, autopct='%1.1f%%',
        colors=sns.color_palette('Blues_r', len(seg_summary)))
plt.title('Share of Total Revenue by RFM Segment')
plt.tight_layout()
plt.show()


## Retention opportunity: At Risk vs. Champions

In [ ]:
at_risk_value = rfm.loc[rfm['segment']=='At Risk', 'monetary'].sum()
champion_value = rfm.loc[rfm['segment']=='Champions', 'monetary'].sum()
lost_value = rfm.loc[rfm['segment']=='Lost Customers', 'monetary'].sum()

print(f"At Risk segment historical value:  ${at_risk_value:,.0f} ({(rfm['segment']=='At Risk').sum()} customers)")
print(f"Champions segment historical value: ${champion_value:,.0f} ({(rfm['segment']=='Champions').sum()} customers)")
print(f"Lost Customers historical value:    ${lost_value:,.0f} ({(rfm['segment']=='Lost Customers').sum()} customers)")


**Business action:** At Risk customers have demonstrated purchase intent (they were frequent buyers) but haven't purchased recently — they are the highest-ROI group for a targeted win-back campaign, since acquiring an equivalent amount of *new* revenue would cost more in marketing spend (see marketing CAC/ROAS analysis in `02_eda.ipynb`).

## Cohort Retention Analysis

**Business question:** Once we acquire a customer, how well do we retain them month over month?

In [ ]:
cohort.index = pd.to_datetime(cohort.index).to_period('M')
cohort_display = cohort.iloc[:, :13]  # first 12 months of retention for readability

plt.figure(figsize=(14, 10))
sns.heatmap(cohort_display, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=0.5,
            cbar_kws={'label': 'Retention Rate'})
plt.title('Monthly Cohort Retention (% of cohort still purchasing, Month 0-12)')
plt.xlabel('Months Since Signup')
plt.ylabel('Signup Cohort (Month)')
plt.tight_layout()
plt.show()


In [ ]:
avg_retention_by_period = cohort.mean(axis=0)
avg_retention_by_period.iloc[:13].plot(kind='bar', color='#1a3d5c', title='Average Retention Curve (all cohorts)')
plt.xlabel('Months Since Signup')
plt.ylabel('Average Retention Rate')
plt.tight_layout()
plt.show()


**Insight:** Retention naturally drops off sharply after month 0 (the signup/first-purchase month is definitionally 100%) and stabilizes at a lower plateau — this plateau is the more meaningful long-term retention figure than month-1 retention alone. Compare the plateau level against the 67.6% overall repeat-purchase rate calculated in SQL (`sql/03_customer_analysis.sql`) — cohort retention is stricter because it requires purchasing *in that specific month*, not just at any point.

**Recommendation:** Use the plateau retention rate as the baseline against which future lifecycle-marketing experiments (win-back emails, loyalty programs) are measured — any initiative should be evaluated by whether it lifts this plateau, not just short-term month-1 numbers.


## Summary

- Revenue is concentrated in Champions/Loyal segments — differentiated retention investment is justified.
- At Risk customers represent the clearest near-term win-back opportunity based on historical value.
- Cohort retention settles into a plateau after initial drop-off — this plateau is the right long-term retention benchmark.

Next: these segments feed directly into the Power BI Customer Intelligence page (Stage 8) and the business report recommendations (`reports/business_insights.md`).
